In [ ]:
import sys
from pathlib import Path
import json
import sqlite3

# Add project root and src to sys.path to allow imports from src/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from visualizations import (
    plot_loss_history,
    plot_potential,
    plot_wavefunctions,
    plot_energy_spectrum,
    plot_pod_singular_values
)

DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "training_runs.db"

# Try to load the latest run_id if available, otherwise use a default or specified one
run_id = "20260424_205517_246896" # Default
run_id_file = DATA_DIR / "run_id.txt"
if run_id_file.exists():
    run_id = run_id_file.read_text().strip()
    print(f"Using run_id from {run_id_file}: {run_id}")
else:
    print(f"Using default/specified run_id: {run_id}")

RUN_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "runs" / run_id

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

run_row = conn.execute(
    "SELECT * FROM runs WHERE run_id = ?",
    (run_id,),
).fetchone()

if run_row is None:
    raise KeyError(f"Run not found: {run_id}")

metrics_df = pd.read_sql_query(
    """
    SELECT epoch, total_loss, physics_loss, data_loss, smooth_loss, ordered_loss
    FROM metrics
    WHERE run_id = ?
    ORDER BY epoch ASC
    """,
    conn,
    params=(run_id,),
)

print(f"Loaded {len(metrics_df)} epochs of metrics.")
metrics_df.head()

run_meta = dict(run_row)
hyperparams = json.loads(run_meta["hyperparams"])

# Plot loss history
fig_loss = plot_loss_history(
    epochs=metrics_df["epoch"].values,
    total=metrics_df["total_loss"].values,
    physics=metrics_df["physics_loss"].values,
    data=metrics_df["data_loss"].values,
    smooth=metrics_df["smooth_loss"].values,
    ordered=metrics_df["ordered_loss"].values,
    lambdas=hyperparams["lambdas"]
)
plt.show()

In [ ]:
import numpy as np
import torch

# Load artifacts
diagnostics = np.load(RUN_ARTIFACTS_DIR / "diagnostics.npz")
ground_truth = torch.load(RUN_ARTIFACTS_DIR / "ground_truth.pt")

x = ground_truth['x']
V_true = ground_truth['V_true']
psi_true = ground_truth['psi_true']
E_true = ground_truth['E_true']

V_learned = torch.from_numpy(diagnostics['V_learned'])
psi_learned = [torch.from_numpy(p) for p in diagnostics['psi_learned']]
E_learned = torch.from_numpy(diagnostics['E_learned'])
pod_singular_values = torch.from_numpy(diagnostics['pod_singular_values'])

# Visualization 1: Potential
plot_potential(x, V_true, V_learned, hyperparams['lambdas'])
plt.show()

# Visualization 2: Wavefunctions
plot_wavefunctions(x, psi_true, psi_learned)
plt.show()

# Visualization 3: Energy Spectrum
plot_energy_spectrum(E_true, E_learned)
plt.show()

# Visualization 4: POD Singular Values
plot_pod_singular_values(pod_singular_values)
plt.show()